# Customer Churn — Exploratory Data Analysis

This notebook explores the `customer_churn.csv` dataset (500 rows × 9 columns) to understand feature distributions, class balance, and relationships before model development.

**Sections**
1. Load & inspect
2. Target distribution (class imbalance)
3. Numerical feature distributions
4. Categorical feature breakdown vs. churn
5. Correlation analysis
6. Key findings → feature engineering decisions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

df = pd.read_csv("../data/customer_churn.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()
print("\nMissing values per column:")
print(df.isnull().sum())

## 2. Target Distribution

In [ ]:
churn_counts = df["Churn"].value_counts()
churn_rate = df["Churn"].mean()
print(f"Churn rate: {churn_rate:.1%}")
print(churn_counts)

fig, ax = plt.subplots(figsize=(6, 5))
sns.countplot(data=df, x="Churn", ax=ax, palette=["#4C72B0", "#DD8452"])
ax.set_xticklabels(["No Churn", "Churn"])
ax.set_title(f"Target Class Distribution (Churn Rate = {churn_rate:.1%})")
plt.show()

# NOTE: ~10.6% churn rate confirms significant class imbalance.
# This is why the preprocessing pipeline applies SMOTE oversampling
# and the model uses class-weighted loss during training.

## 3. Numerical Feature Distributions

In [ ]:
numeric_cols = ["Tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, numeric_cols):
    sns.histplot(data=df, x=col, hue="Churn", kde=True, ax=ax, bins=25, palette=["#4C72B0", "#DD8452"])
    ax.set_title(col)
plt.tight_layout()
plt.show()

df[numeric_cols].describe()

## 4. Categorical Features vs. Churn

In [ ]:
categorical_cols = ["Contract", "PaymentMethod", "PaperlessBilling", "SeniorCitizen"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flatten(), categorical_cols):
    churn_by_cat = df.groupby(col)["Churn"].mean().sort_values(ascending=False)
    churn_by_cat.plot(kind="bar", ax=ax, color="#C44E52")
    ax.set_ylabel("Churn Rate")
    ax.set_title(f"Churn Rate by {col}")
    ax.axhline(churn_rate, color="black", linestyle="--", alpha=0.5, label="Overall rate")
    ax.legend()
plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
encoded = df.copy()
for col in ["Contract", "PaymentMethod", "PaperlessBilling"]:
    encoded[col] = encoded[col].astype("category").cat.codes

corr_cols = ["Tenure", "MonthlyCharges", "TotalCharges", "Contract",
             "PaymentMethod", "PaperlessBilling", "SeniorCitizen", "Churn"]
corr = encoded[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## 6. Key Findings

- **Class imbalance**: churn rate is ~10–11%, confirming the need for SMOTE + class weighting in the training pipeline.
- **Tenure**: shorter-tenure customers show a visibly higher churn rate — consistent with the `TenureBucket` engineered feature in `data_preprocessing.py`.
- **Contract type**: month-to-month customers churn at a meaningfully higher rate than one/two-year contracts, since they face no penalty for leaving.
- **Payment method**: electronic check users show elevated churn versus automatic payment methods (bank transfer / credit card), likely correlating with lower engagement/commitment.
- **MonthlyCharges**: high-paying customers churn somewhat more — informs the `HighChargeFlag` engineered feature.
- **Correlations are modest** (no single feature dominates), which supports using a neural network capable of learning interactions rather than relying on a single strong linear predictor.